This is a p good model for maybe the top end analysis. Seems to be doing pretty well

In [118]:
from transformers import pipeline

# Load the classification pipeline with the specified model
pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis", return_all_scores=False)

# Classify a new sentence
good_sentence = "I think shawn is a great guy"
bad_sentnece = "abood has been really shitty to me lately"
sentence = 'I think Shawn is a great guy but Abood has been really shitty to me lately'
result = pipe(bad_sentnece)

# Print the result
print(result)

[{'label': 'Negative', 'score': 0.5759602189064026}]


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\pipelines\text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


Interest model: https://huggingface.co/firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3

Defo needs more testing, but interesting. It generalizes the whole entry tho. This will need some way to parse sentences which is weird. Maybe this can be used as an over-the-top model? idk

In [33]:
# Requires: librosa
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
import librosa
import torch
import numpy as np

model_id = "firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3"
model = AutoModelForAudioClassification.from_pretrained(model_id)

feature_extractor = AutoFeatureExtractor.from_pretrained(model_id, do_normalize=True)
id2label = model.config.id2label

def preprocess_audio(audio_path, feature_extractor, max_duration=30.0):
    audio_array, sampling_rate = librosa.load(audio_path, sr=None)
    
    max_length = int(feature_extractor.sampling_rate * max_duration)
    if len(audio_array) > max_length:
        audio_array = audio_array[:max_length]
    else:
        audio_array = np.pad(audio_array, (0, max_length - len(audio_array)))

    inputs = feature_extractor(
        audio_array,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=max_length,
        truncation=True,
        return_tensors="pt",
    )
    return inputs

def predict_emotion(audio_path, model, feature_extractor, id2label, max_duration=30.0):
    inputs = preprocess_audio(audio_path, feature_extractor, max_duration)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_id = torch.argmax(logits, dim=-1).item()
    predicted_label = id2label[predicted_id]
    
    return predicted_label

audio_path = r"C:\Users\User\projects\MindLine\testing_models\test_audios\Test3.wav"

predicted_emotion = predict_emotion(audio_path, model, feature_extractor, id2label)
print(f"Predicted Emotion: {predicted_emotion}")


Predicted Emotion: angry


Sarcasm detector cuz sarcasm is still huge

This one below is kinda mid

In [8]:
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
import string

def preprocess_data(text: str) -> str:
   return text.lower().translate(str.maketrans("", "", string.punctuation)).strip()

MODEL_PATH = "helinivan/english-sarcasm-detector"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

text = "Yeah, I love the pain"
tokenized_text = tokenizer([preprocess_data(text)], padding=True, truncation=True, max_length=256, return_tensors="pt")
output = model(**tokenized_text)
probs = output.logits.softmax(dim=-1).tolist()[0]
confidence = max(probs)
prediction = probs.index(confidence)
results = {"is_sarcastic": prediction, "confidence": confidence}
print(results)


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


{'is_sarcastic': 0, 'confidence': 0.896741509437561}


I think we found the one for emotions gang LESGO

In [120]:
from transformers import pipeline

classifier = pipeline(task="text-classification", model="SamLowe/roberta-base-go_emotions", top_k=None)

sentences = [bad_sentnece] 

model_outputs = classifier(sentences)
model_outputs[0]
# produces a list of dicts for each of the labels


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[{'label': 'annoyance', 'score': 0.4621132016181946},
 {'label': 'disappointment', 'score': 0.3438815474510193},
 {'label': 'disgust', 'score': 0.12094248831272125},
 {'label': 'anger', 'score': 0.08203191310167313},
 {'label': 'disapproval', 'score': 0.06776023656129837},
 {'label': 'neutral', 'score': 0.05365602672100067},
 {'label': 'sadness', 'score': 0.04677429050207138},
 {'label': 'approval', 'score': 0.009553383104503155},
 {'label': 'embarrassment', 'score': 0.00941687636077404},
 {'label': 'realization', 'score': 0.008362304419279099},
 {'label': 'desire', 'score': 0.0036978404968976974},
 {'label': 'optimism', 'score': 0.0035749583039432764},
 {'label': 'fear', 'score': 0.0029588225297629833},
 {'label': 'admiration', 'score': 0.0024782842956483364},
 {'label': 'love', 'score': 0.0023896032944321632},
 {'label': 'gratitude', 'score': 0.001833559712395072},
 {'label': 'caring', 'score': 0.001797381672076881},
 {'label': 'nervousness', 'score': 0.001745060901157558},
 {'label'

Testing out the entity detection options

In [61]:
import spacy

sentences = [
    'Today was a great day, man—like I answered really well in all my classes and even got all my Calculus-II homework done on time.', 
    'On top of that, I hit it off with this guy named Shawn and had a lot of fun.', 
    'I think Shawn is a great dude, and Shawn and I would make amazing friends—like, how can this day get any better?'
]

nlp = spacy.load("en_core_web_sm")

for text in sentences:
    print('----')
    doc = nlp(text)
    
    # Iterate through every individual word/token
    for token in doc:
        if token.pos_ in ["PROPN", "NOUN"]:
            print(f"{token.text: <15} | Tag: {token.pos_}")

----
Today           | Tag: NOUN
day             | Tag: NOUN
man             | Tag: NOUN
classes         | Tag: NOUN
Calculus        | Tag: PROPN
II              | Tag: PROPN
homework        | Tag: NOUN
time            | Tag: NOUN
----
top             | Tag: NOUN
guy             | Tag: NOUN
Shawn           | Tag: PROPN
lot             | Tag: NOUN
fun             | Tag: NOUN
----
Shawn           | Tag: PROPN
dude            | Tag: NOUN
Shawn           | Tag: PROPN
friends         | Tag: NOUN
day             | Tag: NOUN


In [86]:
import spacy

nlp = spacy.load("en_core_web_sm")

sentences = [
    'I think Shawn is a great guy, on the other hand Abood has been really shitty to me lately'
]

for text in sentences:
    print(f"\n--- Sentence Analysis ---")
    doc = nlp(text)
    
    print(f"{'Token':<12} | {'Universal POS':<15} | {'Fine-Grained Tag'}")
    print("-" * 50)
    
    for token in doc:
        print(f"{token.text:<12} | {token.pos_:<15} | {token.tag_}")


--- Sentence Analysis ---
Token        | Universal POS   | Fine-Grained Tag
--------------------------------------------------
I            | PRON            | PRP
think        | VERB            | VBP
Shawn        | PROPN           | NNP
is           | AUX             | VBZ
a            | DET             | DT
great        | ADJ             | JJ
guy          | NOUN            | NN
,            | PUNCT           | ,
on           | ADP             | IN
the          | DET             | DT
other        | ADJ             | JJ
hand         | NOUN            | NN
Abood        | PROPN           | NNP
has          | AUX             | VBZ
been         | AUX             | VBN
really       | ADV             | RB
shitty       | ADJ             | JJ
to           | ADP             | IN
me           | PRON            | PRP
lately       | ADV             | RB


In [115]:
import spacy

nlp = spacy.load("en_core_web_sm")


sentence = 'I think Shawn is a great guy, and Abood has been a really shitty person to me lately'


adjectives = []

doc = nlp(sentence)

for token in doc:
    if token.pos_ in ['ADJ']:
        target = token.head
        print(token.text, '-', token.head)

        print("Siblings: ")
        for sibling in target.children:
            print(sibling)
            for siblingling in sibling.children:
                print(siblingling)
        print('---')


# pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis", return_all_scores=False)

# results = []
# for ad in adjectives:
#     results.append(pipe(ad))

# results

great - guy
Siblings: 
a
great
---
shitty - person
Siblings: 
a
shitty
really
---


In [ ]:
import spacy
from transformers import pipeline

nlp = spacy.load("en_core_web_sm")
sentence = 'I think Shawn is great guy, and Abood has been a really shitty person to me lately'
doc = nlp(sentence)

# Initialize sentiment pipe
pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")

extracted_relations = []

for token in doc:
    if token.pos_ == "ADJ":
        target = token.head
        
        # find the Proper Noun (Shawn) it relates to.
        final_target = target.text
        
        # Look for the subject of the clause this adjective belongs to
        # We look for the 'nsubj' (nominal subject) in the same sentence segment
        for sibling in target.children:
            if sibling.dep_ == "nsubj" or sibling.pos_ == "PROPN":
                final_target = sibling.text
                break
        
        # If the head itself is a Proper Noun (like "Shawn is great")
        if target.pos_ == "PROPN":
            final_target = target.text

        sentiment = pipe(token.text)[0]
        
        extracted_relations.append({
            "target": final_target,
            "adjective": token.text,
            "sentiment": sentiment['label']
        })

for rel in extracted_relations:
    print(f"Target: {rel['target']} | Feeling: {rel['adjective']} ({rel['sentiment']})")

Target: guy | Feeling: great (Positive)
Target: person | Feeling: shitty (Negative)
